In [1]:
import sys

sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/utils")
sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/database")

In [2]:
# Base packages -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import logging
import re
import ast
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook

# Paraellel Processing -----------------------------------------------------------------------
import dask
from dask import distributed, dataframe as dd
from dask.diagnostics import ProgressBar

# Model work --------------------------------------------------------------------------------
from transformers import AutoModel, AutoTokenizer
import torch


In [3]:
# hide warnings

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Disable TensorFlow INFO and WARNING messages
import tensorflow as tf
tf.get_logger().setLevel('ERROR')  # Disable TensorFlow ERROR messages (optional)

import warnings
# Filter out Dask client warnings
warnings.filterwarnings("ignore")


In [4]:
import chromaDB as cbd # specter_ef, create_text_splitter, create_chroma_client
from db_tools import db_connection
from utils import import_config

config, keys = import_config()

# Chroma setup --------------------------------------------------------------------------------
chroma_server_host = "3.88.178.230"

chroma_client = cbd.create_chroma_client(chroma_server_host)

# Working Collection
collection = chroma_client.get_or_create_collection("fulltext_specter", embedding_function=specter_embeder)

collection.count()
# Modelish stuff --------------------------------------------------------------------------------

model = AutoModel.from_pretrained("allenai/specter")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter")

# Create embeddings function with specter model
specter_embeder = cbd.specter_ef(model, tokenizer)

specter_embeder.create_text_splitter()


Nanosecond heartbeat on server 1690456465963927683000
[Collection(name=langchain_store), Collection(name=abstracts), Collection(name=fulltext), Collection(name=specter_abstracts)]


In [5]:
# Create dask cluster
dask.config.set(scheduler='processes')  # overwrite default with multiprocessing scheduler

cluster = distributed.LocalCluster(name='local', n_workers=3, memory_limit = '8GiB', threads_per_worker=1)  # Launches a scheduler and workers locally , ip = '127.0.0.2', port = '8788'

client = distributed.Client(cluster)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:34593/status,
Dashboard: http://127.0.0.1:34593/status,Workers: 3
Total threads: 3,Total memory: 24.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37069,Workers: 3
Dashboard: http://127.0.0.1:34593/status,Total threads: 3
Started: Just now,Total memory: 24.00 GiB
Comm: tcp://127.0.0.1:41453,Total threads: 1
Dashboard: http://127.0.0.1:34177/status,Memory: 8.00 GiB
Nanny: tcp://127.0.0.1:39065,


In [7]:
def remove_exisiting(df, collection):
    """ Remove existing ids from dataframe """
    # get unique ids from collection that are the corpus ids
    existing_ids = np.unique(pd.DataFrame(collection.get(include = ['metadatas']))['ids'].str.split('-',expand=True).rename(columns={0:'id',1:'part'})['id'].tolist()).tolist()
    print('N existing ids: ',len(existing_ids))
    print(df.shape)
    
    df = df[~df['corpusId'].astype(str).isin(existing_ids)]
    
    print(df.shape)
    
    return df

In [8]:
fulltext = dd.read_parquet('/home/ubuntu/work/data/fulltext_parquets/*', engine='pyarrow')

In [9]:
fulltext = fulltext.map_partitions(pd.DataFrame.rename, columns={'id': 'corpusid', 'text': 'documents'})

In [10]:
fulltext = fulltext.map_partitions(remove_exisiting, collection=collection, pure = False, meta = fulltext)

In [11]:
# fulltext = fulltext.persist()

Turn partition into dictionary

Functionalized

In [ ]:
def create_ids_metadatas(corpusid, num_chunks): 
    """ Create ids for each chunk """
    
    ids = [f"{corpusid}-{i}" for i in range(num_chunks)]
    
    metadatas = [{'corpusid': corpusid, 'chunk':i} for i in range(num_chunks)]
    
    return ids, metadatas

In [ ]:
def process_dict(d):
    
    corpusid = d['corpusId']
    
    documents = dask.delayed(specter_embeder.split_text)(d['documents'])
    
    ids, metadatas = dask.delayed(create_ids_metadatas)(corpusid, len(documents))
    
    embeddings = dask.delayed(specter_embeder.embed_documents)(documents)

    new_dict = {
        'ids': ids,
        'documents': documents, 
        'metadatas': metadatas,
        'embeddings': embeddings
    }

    try: 
        collection.upsert(**new_dict)
        return True
    except: 
        return False


# Dask functions

# Add documents to collection

- map ddf partitions
- apply function over 
- convert row to dictionary
- process with `process_dict`

In [ ]:
# client.restart()

In [ ]:
# ab_tester = fulltext.iloc[:10000, :]
n_partitions = len(fulltext) // 100

print(n_partitions)

fulltext_dict = fulltext.map_partitions(pd.DataFrame.to_dict('records'))

In [ ]:
partition_size = 10

for i in tqdm_notebook(range(len(chunked_abstracts)), desc='chunk'): 
    
    dict_chunk = chunked_abstracts[i]
    
    # Register the ProgressBar with Dask
    # pbar.register()
    
    # dict_chunk = client.scatter(dict_chunk, broadcast=True)

    futures = [dask.delayed(process_dict)(d) for d in dict_chunk]
    
    for j in tqdm_notebook(range(len(futures_split)), leave = False, desc='partition'):
        
        fs = futures_split[j]
        
        results = dask.compute(*fs, traverse=False)
        
        del results
        
    # After your Dask computation is done, remove the ProgressBar
    # pbar.unregister()
        
    del dict_chunk, futures, futures_split